## Robustness Decomposition + Bootstrap (Unified)

Merges `robustness_decomp.ipynb`, `robustness_decomp_nadavVersion.ipynb`, and
`robustness_decomp_bootstrap_nadavVersion.ipynb`. Run top-to-bottom to:

1. Fit the FE model and augment data,
2. Run robustness specs (score range, rewrite subsets, drop k=1, GPT baseline),
3. Compute essay-clustered bootstraps for each spec,
4. Compute GPT baseline premium-based decomposition (alternative neutral approach),
5. Write tables and a stacked-bar figure to `../../output/robustness_decomp/`.

Adjust `BOOT_REPS` for faster smoke tests.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed
from pyfixest.estimation import feols

sns.set_style("whitegrid")

In [ ]:
BOOT_REPS = 500
RNG_SEED = 42

SAT_PATH = "../../data/results/sat_full/data_sat_scored.csv"
NEUTRAL_PATH = "../../data/results/no_desc/no_desc_scored.csv"

OUTPUT_DIR = "../../output/robustness_decomp/"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")

for path in [OUTPUT_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(path, exist_ok=True)

rng = np.random.default_rng(RNG_SEED)

### Helper functions

In [ ]:
def mean_mask(s: pd.Series, mask: pd.Series) -> float:
    s2 = s[mask & s.notna()]
    return float(s2.mean()) if len(s2) else np.nan


def fit_fe_and_augment(df_in: pd.DataFrame) -> pd.DataFrame:
    """Fit two-way FE (essay + k) on rewrites only (k!=0) and map FEs back."""
    df = df_in.sort_values(["essay_id", "k"]).reset_index(drop=True).copy()
    res = feols("score_high_full ~ 1 | essay_id + k", data=df[df["k"] != 0])
    fes = res.fixef()
    df["fe_essay"] = df["essay_id"].map(fes["C(essay_id)"])
    df["fe_k"] = df["k"].map(fes["C(k)"])
    df["u"] = df["score_high_full"] - df["fe_essay"]
    df["diff"] = df["score_high_full"] - df["score_low_full"]
    return df

In [ ]:
def compute_decomposition_point_estimates(df_in: pd.DataFrame):
    df_ = df_in.copy()

    # --- A) total gap (k==0)
    mA = (df_["low_SES"] == 0) & (df_["k"] == 0) & df_["score_high_full"].notna()
    mB = (df_["low_SES"] == 1) & (df_["k"] == 0) & df_["score_low_full"].notna()
    tmp_vals = pd.Series(np.nan, index=df_.index)
    tmp_vals[mA] = df_.loc[mA, "score_high_full"].astype(float)
    tmp_vals[mB] = df_.loc[mB, "score_low_full"].astype(float)

    group = pd.Series(np.nan, index=df_.index)
    group[mA] = 0
    group[mB] = 1
    m0 = group == 0
    m1 = group == 1

    mean0 = mean_mask(tmp_vals, m0)
    mean1 = mean_mask(tmp_vals, m1)
    total_gap = mean0 - mean1

    # --- B) content gap (k==0), use essay FE
    mK0 = df_["k"] == 0
    content_mean0 = mean_mask(df_["fe_essay"], mK0 & (df_["low_SES"] == 0))
    content_mean1 = mean_mask(df_["fe_essay"], mK0 & (df_["low_SES"] == 1))
    content_gap = content_mean0 - content_mean1

    # --- C) style gap via residual u = score_high_full - fe_essay
    u_mean0 = mean_mask(df_["u"], mK0 & (df_["low_SES"] == 0))
    u_mean1 = mean_mask(df_["u"], mK0 & (df_["low_SES"] == 1))
    style_gap = u_mean0 - u_mean1

    # --- D) ranker-tilting (low-SES originals at k==0)
    m_tilt = (df_["low_SES"] == 1) & (df_["k"] == 0)
    others_gap = mean_mask(df_["score_high_full"] - df_["score_low_full"], m_tilt)

    return {
        "total_gap": total_gap,
        "content_gap": content_gap,
        "style_gap": style_gap,
        "others_gap": others_gap,
        "share_content": content_gap / total_gap if pd.notna(total_gap) and total_gap != 0 else np.nan,
        "share_style": style_gap / total_gap if pd.notna(total_gap) and total_gap != 0 else np.nan,
        "share_others": others_gap / total_gap if pd.notna(total_gap) and total_gap != 0 else np.nan,
    }

In [ ]:
def bootstrap_decomposition(df_in: pd.DataFrame, B: int = BOOT_REPS,
                            seed: int = RNG_SEED, desc: str = "Bootstrap reps"):
    df_base = df_in.sort_values(["essay_id", "k"]).reset_index(drop=True).copy()
    point_df = fit_fe_and_augment(df_base)

    essay_to_idx = df_base.groupby("essay_id").indices
    essay_ids = np.array(list(essay_to_idx.keys()))
    n_clusters = len(essay_ids)
    if n_clusters < 2:
        raise ValueError("Need at least two essays for clustered bootstrap.")

    rng_local = np.random.default_rng(seed)
    seeds = rng_local.integers(0, 2 ** 32 - 1, size=B, dtype=np.uint64)

    def one_rep(s: int) -> dict:
        rng_rep = np.random.default_rng(s)
        sampled = rng_rep.choice(essay_ids, size=n_clusters, replace=True)
        idx = np.concatenate([essay_to_idx[eid] for eid in sampled])
        df_b = df_base.iloc[idx].copy()
        df_b = fit_fe_and_augment(df_b)
        return compute_decomposition_point_estimates(df_b)

    results = Parallel(n_jobs=-1, backend="loky")(
        delayed(one_rep)(int(s)) for s in tqdm(seeds, desc=desc)
    )

    boot_df_stats = pd.DataFrame(results)
    ci_bounds = {
        col: tuple(np.nanpercentile(boot_df_stats[col], [2.5, 97.5]))
        for col in boot_df_stats.columns
    }

    point_est = compute_decomposition_point_estimates(point_df)
    return boot_df_stats, ci_bounds, point_est, point_df

In [ ]:
def run_scenario(label: str, df_source: pd.DataFrame, seed: int, save_prefix: str):
    boot_df, ci_bounds, point_est, point_df = bootstrap_decomposition(
        df_source, B=BOOT_REPS, seed=seed, desc=f"{label} bootstrap"
    )

    point_df.to_csv(os.path.join(OUTPUT_DIR, f"{save_prefix}_rows.csv"), index=False)
    boot_df.to_csv(os.path.join(OUTPUT_DIR, f"{save_prefix}_bootstrap.csv"), index=False)

    stats_order = [
        "total_gap", "content_gap", "style_gap", "others_gap",
        "share_content", "share_style", "share_others",
    ]

    rows = []
    for stat in stats_order:
        lo, hi = ci_bounds[stat]
        se = boot_df[stat].std(ddof=1)
        rows.append({
            "scenario": label,
            "stat": stat,
            "point_estimate": round(point_est[stat], 3),
            "std_error": round(se, 3) if pd.notna(se) else np.nan,
            "ci_2.5": round(lo, 3),
            "ci_97.5": round(hi, 3),
        })

    shares = {
        "scenario": label,
        "content": point_est["share_content"],
        "style": point_est["share_style"],
        "others": point_est["share_others"],
    }

    return {
        "label": label,
        "point": point_est,
        "ci": ci_bounds,
        "rows": rows,
        "shares": shares,
        "boot_df": boot_df,
        "point_df": point_df,
    }

### Load data

In [ ]:
df_sat_raw = pd.read_csv(SAT_PATH, encoding="ISO-8859-1", index_col=0)
df_sat_raw = df_sat_raw.sort_values(["essay_id", "k"]).reset_index(drop=True)

df_neutral_raw = pd.read_csv(NEUTRAL_PATH, encoding="ISO-8859-1", index_col=0)
df_neutral_raw = df_neutral_raw.sort_values(["essay_id", "k"]).reset_index(drop=True)

print(f"SAT rows: {len(df_sat_raw):,} | Neutral rows: {len(df_neutral_raw):,}")

### Scenario builders

In [ ]:
def keep_score_two_to_five(df: pd.DataFrame) -> pd.DataFrame:
    """Score 2-5 restriction with |k - true_score| <= 1 filter."""
    df = df.copy()
    df = df[(df["true_score"] <= 5) & (df["true_score"] >= 2)]
    mask = (np.abs(df["k"] - df["true_score"]) <= 1) | (df["k"] == 0)
    return df[mask]


def keep_rewrites_1_2_5_6(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["k"].isin([0, 1, 2, 5, 6])].copy()


def drop_k1_rewrites(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["k"] != 1].copy()

### Run FE decompositions + bootstraps

In [ ]:
scenario_results = {}
summary_rows = []
share_rows = []

sat_scenarios = [
    ("sat_baseline", "SAT (baseline)", lambda d: d.copy(), 42),
    ("sat_score25", "Score 2\u20135, |\u0394| \u2264 1", keep_score_two_to_five, 43),
    ("sat_rewrites_1256", "Rewrites {1,2,5,6} only", keep_rewrites_1_2_5_6, 44),
    ("sat_drop_k1", "Drop k=1 rewrites", drop_k1_rewrites, 45),
]

for key, label, builder, seed in sat_scenarios:
    df_variant = builder(df_sat_raw)
    res = run_scenario(label, df_variant, seed=seed, save_prefix=key)
    scenario_results[key] = res
    summary_rows.extend(res["rows"])
    share_rows.append(res["shares"])

# Neutral baseline (FE-based)
neutral_res = run_scenario(
    "Baseline GPT", df_neutral_raw.copy(), seed=99, save_prefix="neutral_baseline"
)
scenario_results["neutral_baseline"] = neutral_res
summary_rows.extend(neutral_res["rows"])
share_rows.append(neutral_res["shares"])

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(TABLE_DIR, "robustness_bootstrap_summary_long.csv"), index=False)
summary_df

### Pivoted summary table (point \u00b1 SE)

In [ ]:
STAT_LABELS = {
    "total_gap": "Total Gap",
    "content_gap": "Content",
    "style_gap": "Style",
    "others_gap": "Other",
    "share_content": "Share Content",
    "share_style": "Share Style",
    "share_others": "Share Other",
}
STAT_ORDER = list(STAT_LABELS.keys())

SCENARIO_ORDER = [
    "SAT (baseline)",
    "Score 2\u20135, |\u0394| \u2264 1",
    "Rewrites {1,2,5,6} only",
    "Drop k=1 rewrites",
    "Baseline GPT",
]


def format_est_se(row):
    pe = row["point_estimate"]
    se = row.get("std_error", np.nan)
    if pd.isna(pe):
        return ""
    if pd.isna(se):
        return f"{pe:.3f}"
    return f"\\shortstack{{{pe:.3f}\\\\({se:.3f})}}"


summary_formatted = summary_df.copy()
summary_formatted["stat"] = summary_formatted["stat"].map(lambda x: STAT_LABELS.get(x, x))
summary_formatted["estimate_se"] = summary_formatted.apply(format_est_se, axis=1)

summary_pivot = summary_formatted.pivot(index="scenario", columns="stat", values="estimate_se")
summary_pivot = summary_pivot.reindex(columns=[STAT_LABELS[s] for s in STAT_ORDER if STAT_LABELS[s] in summary_pivot.columns])
summary_pivot = summary_pivot.reindex([s for s in SCENARIO_ORDER if s in summary_pivot.index])
summary_pivot = summary_pivot.rename_axis(index=None, columns=None)

summary_pivot.to_csv(os.path.join(TABLE_DIR, "robustness_bootstrap_summary_pivot.csv"))
summary_pivot.to_latex(os.path.join(TABLE_DIR, "robustness_bootstrap_summary_pivot.tex"), escape=False)
summary_pivot

### GPT Baseline: Premium-Based Decomposition

An alternative decomposition of the neutral-rewrite data that directly computes
content, style-premium, and scorer components without fixed effects.
Decomposes the observed gap into:
- **Content**: gap in neutral-rewrite scores across SES groups
- **Style premium**: differential style premium (high-SES premium minus low-SES premium)
- **Scorer**: difference in how low-SES originals are scored by the two scorers

In [ ]:
df_neutral = df_neutral_raw.copy()

mask_H = df_neutral["low_SES"] == 0
mask_L = df_neutral["low_SES"] == 1
mask_orig = df_neutral["k"] == 0
mask_neu = df_neutral["k"] != 0

# Means under scorer S(H)
mu_H_orig = mean_mask(df_neutral["score_high_full"], mask_H & mask_orig)
mu_L_orig = mean_mask(df_neutral["score_high_full"], mask_L & mask_orig)
mu_H_neu = mean_mask(df_neutral["score_high_full"], mask_H & mask_neu)
mu_L_neu = mean_mask(df_neutral["score_high_full"], mask_L & mask_neu)

# Under scorer S(L), originals only
mu_L_orig_Lscorer = mean_mask(df_neutral["score_low_full"], mask_L & mask_orig)

means = {
    "mu_H_orig": mu_H_orig,
    "mu_L_orig": mu_L_orig,
    "mu_H_neu": mu_H_neu,
    "mu_L_neu": mu_L_neu,
    "mu_L_orig_Lscorer": mu_L_orig_Lscorer,
}
print("Group means:")
means

In [ ]:
total_gap = mu_H_orig - mu_L_orig_Lscorer  # observed gap

content_gap = mu_H_neu - mu_L_neu

highPremium = mu_H_orig - mu_H_neu
lowPremium = mu_L_orig - mu_L_neu
style_gap = highPremium - lowPremium

scorer_gap = mu_L_orig - mu_L_orig_Lscorer

gpt_decomp = {
    "total_gap": total_gap,
    "content_gap": content_gap,
    "style_gap": style_gap,
    "scorer_gap": scorer_gap,
    "share_content": content_gap / total_gap if total_gap else np.nan,
    "share_style": style_gap / total_gap if total_gap else np.nan,
    "share_scorer": scorer_gap / total_gap if total_gap else np.nan,
    "high_premium": highPremium / total_gap if total_gap else np.nan,
    "low_premium": lowPremium / total_gap if total_gap else np.nan,
}
gpt_decomp

In [ ]:
def build_neutral_decomp_df(df_in: pd.DataFrame) -> pd.DataFrame:
    """Build per-essay dataset matching originals to neutral rewrites."""
    df = df_in.copy()

    df_orig = df[df["k"] == 0].copy()
    df_neu = df[df["k"] == 1].copy()

    # Drop grade levels 9.0 and 12.0
    df_orig = df_orig[~df_orig["grade_level"].isin([9.0, 12.0])]
    df_neu = df_neu[df_neu["essay_id"].isin(df_orig["essay_id"])]

    keep_cols = [
        "essay_id", "low_SES", "race_white", "gender_male",
        "prompt_name", "grade_level", "cv_fold",
        "score_high_full", "score_low_full",
    ]

    df_orig = df_orig[keep_cols].rename(columns={
        "score_high_full": "score_high_orig",
        "score_low_full": "score_low_orig",
    })

    df_neu = df_neu[["essay_id", "score_high_full"]].rename(columns={
        "score_high_full": "score_high_neu",
    })

    out = df_orig.merge(df_neu, on="essay_id", how="inner")

    out["content_neu"] = out["score_high_neu"]
    out["style_premium"] = out["score_high_orig"] - out["score_high_neu"]
    out["diff_high_low_orig"] = out["score_high_orig"] - out["score_low_orig"]
    out["k"] = 0

    return out


df_neutral_essay = build_neutral_decomp_df(df_neutral_raw)
df_neutral_essay.head()

### Stacked-bar figure for decomposition shares

In [ ]:
COMP_ORDER = ["content", "style", "others"]
COMP_COLORS = {"content": "#84C7F9", "style": "#C7B6F7", "others": "#B6E3C7"}

shares_df = pd.DataFrame(share_rows).set_index("scenario")[COMP_ORDER]

ax = shares_df.plot(
    kind="bar",
    stacked=True,
    color=[COMP_COLORS[c] for c in COMP_ORDER],
    edgecolor="white",
    linewidth=0.6,
    figsize=(9, 5),
)

ax.set_ylabel("Share of total gap")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.axhline(1, color="gray", lw=0.8, linestyle="--")
plt.xticks(rotation=20, ha="right")
ax.legend(title="Component", loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "robustness_decomposition_shares.pdf"), bbox_inches="tight")
plt.show()